# model-save-state-dict — ex1: save state_dict from rank 0 with barrier

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `model-save-state-dict`. Running the final beacon cell reports progress against the `Distributed: model save state_dict rank-0` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: model save state_dict rank-0` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`model-save-state-dict`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "model-save-state-dict"
DD_SUBTOPIC = "Distributed: model save state_dict rank-0"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.distributed quick refresher

PyTorch's collective-communication library (`torch.distributed`, aliased `dist`) lets multiple processes coordinate over tensors. Each rank runs the same function in its own process; collectives operate in-place on tensors of identical shape across ranks.

**Backends.** `'nccl'` for multi-GPU (ARENA's setup), `'gloo'` for CPU (what these drills use — Colab CPU runtimes have no GPUs).

**Launch pattern.** Each test uses `mp.get_context('fork').Process` so worker fns defined in a notebook cell are picklable. Workers init the group, do their work, push results onto a `manager.Queue`, then destroy the group.

### This drill's atom: save state_dict from rank 0 only
After grad-sync, every rank holds identical parameters. Saving a checkpoint from EVERY rank would (a) waste IO bandwidth, (b) race on the file path. The standard pattern:
```python
if rank == 0:
    t.save(model.state_dict(), checkpoint_path)
dist.barrier()  # other ranks wait for rank 0 to finish
```
**Why the barrier.** Without it, rank 1 might charge ahead into the next epoch before rank 0 finishes writing — fine on its own, but if code later does `if rank == 1: load(checkpoint)` you'll race. `dist.barrier()` makes EVERY rank stop until all ranks reach it.

### Exercise 1 — save state_dict from rank 0 with barrier

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `if rank == 0: save; dist.barrier()` pattern so a single process writes the checkpoint while all other ranks wait at the barrier before continuing.
> Keywords: state_dict, checkpoint, rank-0-only, barrier, side-effect
> ```

**KCs targeted:** `rank0-only-side-effects`, `dist-barrier-synchronization`

Implement `ex1_save_checkpoint_worker(rank, world_size, port, ckpt_path, out_queue)`. Each rank:

1. Inits `gloo`.
2. Builds an identical model (rank-independent for this drill):
   ```python
   model = t.nn.Linear(2, 2, bias=False)
   with t.no_grad():
       model.weight.fill_(7.0)
   ```
3. **Saves the state_dict from rank 0 only, then barriers:**
   ```python
   if rank == 0:
       t.save(model.state_dict(), ckpt_path)
   dist.barrier()
   ```
4. After the barrier, EVERY rank reads back the file to confirm it exists and contains the expected tensor:
   ```python
   sd = t.load(ckpt_path, weights_only=True)
   out_queue.put((rank, sd['weight'].sum().item()))
   ```
5. Destroys process group.

**Expected.** After the barrier, every rank can load `ckpt_path` and the loaded weights sum to `7.0 * 4 = 28.0` (2×2 matrix filled with 7s).

In [ ]:
def ex1_save_checkpoint_worker(rank, world_size, port, ckpt_path, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size,
                            timeout=datetime.timedelta(seconds=20))
    model = t.nn.Linear(2, 2, bias=False)
    with t.no_grad():
        model.weight.fill_(7.0)
    if rank == 0:
        t.save(model.state_dict(), ckpt_path)
    dist.barrier()  # other ranks wait until rank-0 finishes the write
    sd = t.load(ckpt_path, weights_only=True)
    out_queue.put((rank, sd['weight'].sum().item()))
    dist.destroy_process_group()


<details><summary>Solution</summary>

```python
def ex1_save_checkpoint_worker(rank, world_size, port, ckpt_path, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size,
                            timeout=datetime.timedelta(seconds=20))
    model = t.nn.Linear(2, 2, bias=False)
    with t.no_grad():
        model.weight.fill_(7.0)
    if rank == 0:
        t.save(model.state_dict(), ckpt_path)
    dist.barrier()  # other ranks wait until rank-0 finishes the write
    sd = t.load(ckpt_path, weights_only=True)
    out_queue.put((rank, sd['weight'].sum().item()))
    dist.destroy_process_group()
```

**Why rank 0 only.** Disk IO from N ranks to the same path = N concurrent writes, undefined contents, possibly corrupted file. Even when writing to different paths, save time scales with N — wasteful when ranks 1..N-1 hold identical tensors.

**Why the barrier.** Without `dist.barrier()`, rank 1 might reach the next collective op (say, `dist.all_reduce` in the next epoch) before rank 0 finishes saving. Real DDP usually survives this — but ANY code that depends on the file existing (e.g. immediate reload, an external evaluator) will race.

**`weights_only=True` since PyTorch 2.6.** Mandatory for security: `t.load` used to allow arbitrary code execution via pickle, now the default refuses unless you opt in. For state_dicts, `weights_only=True` is always safe.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()